## create invoice_extraction_table

In [0]:

%sql
CREATE TABLE IF NOT EXISTS data_engineering_workshop.agents.invoice_extractions(
  invoice_number  STRING,
  invoice_date    DATE,
  due_date        DATE,
  po_number       STRING,
  payment_terms   STRING,
  source_file     STRING,
  extracted_at    TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

### Create Tracking Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS data_engineering_workshop.agents.processed_invoice_files (
  file_path     STRING NOT NULL,
  processed_at  TIMESTAMP,
  status        STRING
)
USING DELTA;

### Extract Only NEW Files

In [0]:
%sql
INSERT INTO data_engineering_workshop.agents.invoice_extractions

WITH new_files AS (
  SELECT path, content
  FROM read_files(
    '/Volumes/data_engineering_workshop/agents/agentsdocs',
    format => 'binaryFile'
  )
  WHERE path NOT IN (
    SELECT file_path 
    FROM data_engineering_workshop.agents.processed_invoice_files
    WHERE status = 'success'
  )
),
parsed AS (
  SELECT
    path,
    ai_parse_document(
      content,
      map('version', '2.0', 'descriptionElementTypes', '*')
    ) AS parsed
  FROM new_files
),
extracted AS (
  SELECT
    path,
    ai_extract(
      parsed,
      '{
        "invoice_number": {"type":"string","description":"The unique identifier of the invoice"},
        "invoice_date":   {"type":"string","description":"The date the invoice was issued in YYYY-MM-DD format"},
        "due_date":       {"type":"string","description":"The date payment is due in YYYY-MM-DD format"},
        "po_number":      {"type":"string","description":"The purchase order number associated with the invoice"},
        "payment_terms":  {"type":"string","description":"The payment terms specified on the invoice"}
      }',
      options => map('version', '2.0')
    ) AS response
  FROM parsed
  WHERE parsed IS NOT NULL
)

SELECT
  response:response:invoice_number::STRING   AS invoice_number,
  response:response:invoice_date::DATE       AS invoice_date,
  response:response:due_date::DATE           AS due_date,
  response:response:po_number::STRING        AS po_number,
  response:response:payment_terms::STRING    AS payment_terms,
  path                                       AS source_file,
  current_timestamp()                        AS extracted_at
FROM extracted
WHERE response:error_message::STRING IS NULL;

### Mark Processed Files (Prevents Re-processing)

In [0]:
%sql
INSERT INTO data_engineering_workshop.agents.processed_invoice_files

SELECT DISTINCT
  path                AS file_path,
  current_timestamp() AS processed_at,
  'success'           AS status
FROM read_files(
  '/Volumes/data_engineering_workshop/agents/agentsdocs',
  format => 'binaryFile'
)
WHERE path NOT IN (
  SELECT file_path 
  FROM data_engineering_workshop.agents.processed_invoice_files
  WHERE status = 'success'
);